In [1]:
import matplotlib.pyplot as plt
import numpy as np
import os
import sys
import pickle

path = os.getcwd().split(os.sep + 'GUI')[0]
if path not in sys.path:
    print("not here")
    sys.path.append(path)

from neurolib.models.aln import ALNModel
from neurolib.utils import plotFunctions as plotFunc
from neurolib.utils import costFunctions as cost
import neurolib.dashboard.functions as functions
import neurolib.dashboard.data as data
    
# This will reload all imports as soon as the code changes
%load_ext autoreload
%autoreload 2 

#path = os.path.join(os.getcwd(), "plots")

not here


In [2]:
# read case
print(os.getcwd())
case = os.getcwd().split(os.sep)[-1]
print(case[0])

/mnt/antares_raid/home/salfenmoser/neurolib/GUI/current/gui/data/10001
1


### Bistability

In [3]:
aln = ALNModel()
N = aln.params.N

data.set_parameters(aln)

state_vars = aln.state_vars
init_vars = aln.init_vars

##############################################################
def setinit(init_vars_, model):
    state_vars = model.state_vars
    init_vars = model.init_vars
    for iv in range(len(init_vars)):
        for sv in range(len(state_vars)):
            if state_vars[sv] in init_vars[iv]:
                #print("set init vars ", )
                if model.params[init_vars[iv]].ndim == 2:
                    model.params[init_vars[iv]][0,:] = init_vars_[sv]
                else:
                    model.params[init_vars[iv]][0] = init_vars_[sv]
                    
##############################################################               
def setmaxmincontrol(max_c_c, min_c_c, max_c_r, min_c_r):
    import numpy as np
    
    max_cntrl = np.zeros(( 6 ))
    min_cntrl = np.zeros(( 6 ))
    
    max_cntrl[0] = max_c_c
    min_cntrl[0] = min_c_c
    max_cntrl[1] = max_c_c
    min_cntrl[1] = min_c_c
    max_cntrl[2] = max_c_r
    min_cntrl[2] = min_c_r
    max_cntrl[3] = max_c_r
    min_cntrl[3] = min_c_r
    max_cntrl[4] = max_c_r
    min_cntrl[4] = min_c_r
    max_cntrl[5] = max_c_r
    min_cntrl[5] = min_c_r
            
    return max_cntrl, min_cntrl

#####################################################
def getclosest(k_, found_solution, exc, inh, already_tried_):
    import numpy as np
    if len(found_solution) == 0:
        print("no solutions found")
        return -1
    
    start_ind = -1
    for j_ in found_solution:
        if j_ not in already_tried_ and j_ != k_:
            start_ind = j_
            break
            
    if start_ind == -1:
        return -1
        
    min_dist = np.sqrt((exc[k_] - exc[start_ind])**2 + (inh[k_] - inh[start_ind])**2)
    min_i = start_ind
        
    print(found_solution, already_tried_)
        
    if len(found_solution) == len(already_tried_):
        print("already tried all options")
        min_i = -1
        return min_i
    
    for i_ in found_solution:
        if i_ not in already_tried_:
            if i_ != k_ and i_ != min_i:
                dist_ = np.sqrt((exc[k_] - exc[i_])**2 + (inh[k_] - inh[i_])**2)
                if dist_ < min_dist:
                    min_dist = dist_
                    min_i = i_
                    
    if min_i == 0 and 0 in already_tried_:
        return -1
    
    return min_i

In [4]:
##### LOAD BOUNDARIES
data_file = 'bi.pickle'
with open(data_file,'rb') as f:
    load_array= pickle.load(f)
exc = load_array[0]
inh = load_array[1]
print(len(exc))
#plt.scatter(exc, inh)

147


In [5]:
bestControl_init = [None] * len(exc)
bestState_init = [None] * len(exc)
cost_init = [None] * len(exc)
runtime_init = [None] * len(exc)
grad_init = [None] * len(exc)
phi_init = [None] * len(exc)
costnode_init = [None] * len(exc)
weights_init = [None] * len(exc)

conv_init = [[False]*2] * len(exc)

In [6]:
bestControl_0 = [None] * len(exc)
bestState_0 = [None] * len(exc)
cost_0 = [None] * len(exc)
runtime_0 = [None] * len(exc)
grad_0 = [None] * len(exc)
phi_0 = [None] * len(exc)
costnode_0 = [None] * len(exc)
weights_0 = [None] * len(exc)

conv_0 = [[False]*2] * len(exc)

cgv_list = [None, "HS", "FR", "PR", "CD", "LS", "DY", "WYL", "HZ", None]

In [7]:
initVars = [None] * len(exc)
target = [None] * len(exc)
cost_uncontrolled = [None] * len(exc)

In [8]:
dur_pre = 10
dur_post = 10

n_pre = int(np.around(dur_pre/aln.params.dt + 1.,1))
n_post = int(np.around(dur_post/aln.params.dt + 1.,1))

tol = 1e-32
start_step = 10.
c_scheme = np.zeros(( 1,1 ))
c_scheme[0,0] = 1.
u_mat = np.identity(1)
u_scheme = np.array([[1.]])

c_var = [ [0], [1], [0,1]]
p_var = [ [0], [0], [0]]

### CURRENTS
cntrl_vars_0 = [0,1]
prec_vars = [0]

if case[0] == '0':    # low to high
    max_I = [3., -3.]
elif case[0] == '1':
    max_I = [-3., 3.]
    
if case[1] == '0':    # sparsity
    factor_ws = 1.
    factor_we = 0.
elif case[1] == '1':  # energy
    factor_ws = 0.
    factor_we = 1.
    
if case[3] == '0' and case[2] == '0':
    cntrl_vars_init = [0]
elif case[3] == '1' and case[2] == '0':
    cntrl_vars_init = [1]
elif case[3] == '0' and case[2] == '1':
    cntrl_vars_init = [2,4]
elif case[3] == '1' and case[2] == '1':
    cntrl_vars_init = [3,5]

if case[4] == '0':
    dur = 100
    trans_time = 0.8
elif case[4] == '1':
    dur = 400
    trans_time = 0.95
    
maxC = [5., -5., 0.18, 0.]

n_dur = int(np.around(dur/aln.params.dt + 1.,1))
max_cntrl, min_cntrl = setmaxmincontrol(maxC[0], maxC[1], maxC[2], maxC[3])

In [9]:
init_file = 'control_init_' + case + '.pickle'
final_file = 'control_0_' + case + '.pickle'
final_file_1 = 'control_1_' + case + '.pickle'

In [10]:
if os.path.isfile(init_file) :
    print("file found")
    
    with open(init_file,'rb') as f:
        load_array = pickle.load(f)

    bestControl_init = load_array[0]
    bestState_init = load_array[1]
    cost_init = load_array[2]
    runtime_init = load_array[3]
    grad_init = load_array[4]
    phi_init = load_array[5]
    costnode_init = load_array[6]
    weights_init = load_array[7]

file found


In [11]:
i_stepsize = 5
i_range = range(0, len(exc),i_stepsize)
data.set_parameters(aln)

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = 3000.
    
    control0 = aln.getZeroControl()
    control0 = functions.step_control(aln, maxI_ = max_I[0])

    aln.run(control=control0)
    
    target_rates = np.zeros((2))
    target_rates[0] = aln.rates_exc[0,-1] 
    target_rates[1] = aln.rates_inh[0,-1]

    control0 = functions.step_control(aln, maxI_ = max_I[1])
    aln.run(control=control0)

    init_state_vars = np.zeros(( len(state_vars) ))
    for j in range(len(state_vars)):
        if aln.state[state_vars[j]].size == 1:
            init_state_vars[j] = aln.state[state_vars[j]][0]
        else:
            init_state_vars[j] = aln.state[state_vars[j]][0,-1]

    initVars[i] = init_state_vars
    
    aln.params.duration = dur

    target[i] = aln.getZeroTarget()
    target[i][:,0,:] = target_rates[0]
    target[i][:,1,:] = target_rates[1]

-------  0 0.4000000000000001 0.3500000000000001
-------  5 0.4000000000000001 0.40000000000000013
-------  10 0.4250000000000001 0.42500000000000016
-------  15 0.4500000000000001 0.4500000000000002
-------  20 0.4500000000000001 0.4750000000000002
-------  25 0.4250000000000001 0.5000000000000002
-------  30 0.4250000000000001 0.5250000000000002
-------  35 0.5500000000000003 0.5250000000000002
-------  40 0.5250000000000001 0.5500000000000003
-------  45 0.5000000000000002 0.5750000000000003
-------  50 0.47500000000000014 0.6000000000000003
-------  55 0.4250000000000001 0.6250000000000003
-------  60 0.5500000000000003 0.6250000000000003
-------  65 0.5000000000000002 0.6500000000000004
-------  70 0.4500000000000001 0.6750000000000004
-------  75 0.5750000000000002 0.6750000000000004
-------  80 0.5250000000000001 0.7000000000000004
-------  85 0.47500000000000014 0.7250000000000004
-------  90 0.6000000000000003 0.7250000000000004
-------  95 0.5250000000000001 0.750000000000000

In [12]:
data.set_parameters(aln)

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = dur
        
    cost.setParams(1.0, 0.0, 0.0)

##### zero control as input for uncontrolled cost
    setinit(initVars[i], aln)
    control0 = aln.getZeroControl()

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 0

    bestControl_init_, bestState_init_, cost_init_, runtime_init_, grad_init_, phi_init_, costnode_init_ = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    cost_uncontrolled[i] = cost_init_[0]

-------  0 0.4000000000000001 0.3500000000000001
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5902.406479238383
Gradient descend method:  None
RUN  0 , total integrated cost =  5902.406479238383
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5097.289828199723
Gradient descend method:  None
RUN  0 , total integrated cost =  5097.289828199723
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  10 0.4250000000000001 0.42500000000000016
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  9111.456490210901
Gradient descend method:  None
RUN  0 , total integrated cost =  9111.456490210901
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  15 0.4500000000000001 0.4500000000000002

RUN  0 , total integrated cost =  23532.636143093983
Gradient descend method:  None
RUN  0 , total integrated cost =  23532.636143093983
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  140 0.4500000000000001 0.9000000000000006
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  10019.968518582271
Gradient descend method:  None
RUN  0 , total integrated cost =  10019.968518582271
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  145 0.5750000000000002 0.9000000000000006
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  33290.05146687772
Gradient descend method:  None
RUN  0 , total integrated cost =  33290.05146687772
Improved over  0  iterations in  0.0  seconds by  0.0  percent.


In [13]:
i_range_ = []

for i in i_range:
    if type(bestControl_init[i]) == type(None):
        i_range_.append(i)

i_range = np.array(i_range_)
        
print(i_range)

[  5  15  25  45  65  75  85  95 115 125 135 145]


In [14]:
factor_iteration = 20.

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = dur
        
##### zero control as input for uncontrolled cost
    setinit(initVars[i], aln)
    control0 = aln.getZeroControl()

##### initial guess
    weight_ = 10
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    #if prev_i != -1:
    #    control0 = bestControl_init[prev_i][:,:,n_pre-1:-n_post+1]
    
    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int(100 * factor_iteration)
    
    weights_init[i] = cost.getParams()

    bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_init[i][-j] == 0.:
        j += 1
    
    weight_ = 10 * cost_uncontrolled[i] / cost_init[i][-j]
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int(500 * factor_iteration)

    weights_init[i] = cost.getParams()
    
    bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    prev_i = i

-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5097.289828199723
Gradient descend method:  None
RUN  1 , total integrated cost =  27.775589623331495
RUN  2 , total integrated cost =  25.261745079514895
RUN  3 , total integrated cost =  18.988144071372055
RUN  4 , total integrated cost =  17.8239169679096
RUN  5 , total integrated cost =  16.04627432925638
RUN  6 , total integrated cost =  15.275637658261797
RUN  7 , total integrated cost =  14.111498082628268
RUN  8 , total integrated cost =  13.527579154008471
RUN  9 , total integrated cost =  12.718839278428062
RUN  10 , total integrated cost =  12.299602171782807
RUN  11 , total integrated cost =  11.710464609845175
RUN  12 , total integrated cost =  11.374504871398084
RUN  13 , total integrated cost =  10.938322896386781
RUN  14 , total integrated cost =  10.683595116028417
RUN  15 , total integrated cost =  10.356292648901285


ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  411 , total integrated cost =  5.870533941883495
Improved over  411  iterations in  44.56921694241464  seconds by  99.8848302894333  percent.
Problem in initial value trasfer:  Vmean_exc -67.89107561700966 -67.89412797789038
weight =  8682.8385265486
set cost params:  1.0 0.0 8682.8385265486
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5091.938392125415
Gradient descend method:  None
RUN  1 , total integrated cost =  5064.772277911361
RUN  2 , total integrated cost =  5064.672923305512
RUN  3 , total integrated cost =  5064.672792113512
RUN  4 , total integrated cost =  5064.672786933932
RUN  5 , total integrated cost =  5064.672786933926
RUN  6 , total integrated cost =  5064.672786933924


ERROR:root:Problem in initial value trasfer


RUN  7 , total integrated cost =  5064.672786933924
Control only changes marginally.
RUN  7 , total integrated cost =  5064.672786933924
Improved over  7  iterations in  0.7050269339233637  seconds by  0.5354661249173205  percent.
Problem in initial value trasfer:  Vmean_exc -66.85166715325647 -66.87327862402192
-------  15 0.4500000000000001 0.4500000000000002
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  13018.074640346456
Gradient descend method:  None
RUN  1 , total integrated cost =  91.50920686258372
RUN  2 , total integrated cost =  81.13066415394957
RUN  3 , total integrated cost =  61.657956449681016
RUN  4 , total integrated cost =  55.96218457606823
RUN  5 , total integrated cost =  48.28229962499195
RUN  6 , total integrated cost =  45.20480281289532
RUN  7 , total integrated cost =  41.66263592519202
RUN  8 , total integrated cost =  39.83681878315759
RUN  9 , total integrated cost =  37.93477684097143
RUN  10 , tota

...

RUN  12 , total integrated cost =  22949.19659689395
Control only changes marginally.
RUN  12 , total integrated cost =  22949.19659689395
Improved over  12  iterations in  1.0336391981691122  seconds by  1.986351220648288  percent.
Problem in initial value trasfer:  Vmean_exc -59.608612920025635 -59.61722225961443
-------  145 0.5750000000000002 0.9000000000000006
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  33290.05146687772
Gradient descend method:  None
RUN  1 , total integrated cost =  191.5809377818034
RUN  2 , total integrated cost =  122.78712422204399
RUN  3 , total integrated cost =  65.96997354245924
RUN  4 , total integrated cost =  64.33650295769561
RUN  5 , total integrated cost =  63.320582757407074
RUN  6 , total integrated cost =  62.69809996759183
RUN  7 , total integrated cost =  62.24598131603481
RUN  8 , total integrated cost =  61.870319950137045
RUN  9 , total integrated cost =  61.59774668260908
RUN  10 ,

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  304 , total integrated cost =  52.6296826253686
Improved over  304  iterations in  21.634309645742178  seconds by  99.84190567359822  percent.
Problem in initial value trasfer:  Vmean_exc -65.04318076281757 -65.05873726131361
weight =  6325.337681369794
set cost params:  1.0 0.0 6325.337681369794
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  32980.16590480063
Gradient descend method:  None
RUN  1 , total integrated cost =  32013.89922279945
RUN  2 , total integrated cost =  31979.868609734505
RUN  3 , total integrated cost =  31979.86592350629
RUN  4 , total integrated cost =  31979.865007447133
RUN  5 , total integrated cost =  31979.860585917035
RUN  6 , total integrated cost =  31979.835340064627
RUN  7 , total integrated cost =  31979.83165358992
RUN  8 , total integrated cost =  31979.830676425467
RUN  9 , total integrated cost =  31979.824810854938
RUN  10 , total integrated cost =  31979.799811538072
RUN  11 , total

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  22 , total integrated cost =  31978.464823926035
Improved over  22  iterations in  1.7498996928334236  seconds by  3.037283328913716  percent.
Problem in initial value trasfer:  Vmean_exc -57.57918207590453 -57.55982758962617


In [15]:
#plot initial guesses
"""
for i in i_range:
        
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    plotFunc.plot_control_current(aln, [bestControl_init[i]],
        [costnode_init[i]], [weights_init[i]], dur,
        dur_pre, dur_post, initVars[i], target[i], '', filename_ = '', transition_time_ = trans_time,
        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)
"""

'\nfor i in i_range:\n        \n    aln.params.mue_ext_mean = exc[i] * 5.\n    aln.params.mui_ext_mean = inh[i] * 5.\n\n    plotFunc.plot_control_current(aln, [bestControl_init[i]],\n        [costnode_init[i]], [weights_init[i]], dur,\n        dur_pre, dur_post, initVars[i], target[i], \'\', filename_ = \'\', transition_time_ = trans_time,\n        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)\n'

In [16]:
found_solution = []
no_solution = []
last_update = -1
factor_iteration = 20.
already_tried = [ [] for _ in range(len(exc)) ]
i_range = range(0, len(exc),i_stepsize)
i_range_0 = []
i_range_1 = []

print(already_tried, len(already_tried))

for k in range(len(i_range)**2):
    print('------------------------------------------------------------')
    print('--------------------', k)
    print('------------------------------------------------------------')
        
    print("found solution: ", found_solution)
    print("no solution: ", no_solution)
    
    if len(i_range) == len(found_solution) + len(no_solution):
        print("found solution for all parameters")
        break
    
    #if last_update != k-1:
    #    print("no improvement from previous step")
    #    break

    for i in i_range:
        print("------- ", i, exc[i], inh[i])

        if np.abs(bestState_init[i][0,0,-1] - target[i][0,0,-1]) < 0.5 * np.abs(
            bestState_init[i][0,0,-1] - bestState_init[i][0,0,0]):
            if i not in found_solution:
                print("found solution for ", i)
                found_solution.append(i)
                #last_update = k
            if i in no_solution:
                no_solution.pop(no_solution.index(i))
            continue

        #if i not in no_solution:
        #    print("no solution for ", i)
        #    no_solution.append(i)
        
        if i not in i_range_0:
            i_range_0.append(i)
            
        if i not in i_range_1:
            i_range_1.append(i)

        closest_ = getclosest(i, found_solution, exc, inh, already_tried[i])
        print("closest index ", closest_)

        weight_ = 10
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
            
        if i != 0 and closest_ != -1:
            control0 = bestControl_init[closest_][:,:,n_pre-1:-n_post+1]
            if closest_ not in already_tried[i]:
                already_tried[i].append(closest_)
                        
        if closest_ == -1:
            print("all options tried already")
            if i not in no_solution:
                no_solution.append(i)
                continue

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int(100 * factor_iteration)

        weights_init[i] = cost.getParams()
        
        print("precision vars = ", prec_vars)

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)

        j = 1
        while cost_init[i][-j] == 0.:
            j += 1

        weight_ = 10 * cost_uncontrolled[i] / cost_init[i][-j]
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int(500 * factor_iteration)

        weights_init[i] = cost.getParams()

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)

[[], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], []] 147
------------------------------------------------------------
-------------------- 0
------------------------------------------------------------
found solution:  []
no solution:  []
-------  0 0.4000000000000001 0.3500000000000001
found solution for  0
-------  5 0.4000000000000001 0.40000000000000013
found solution for  5
-------  10 0.4250000000000001 0.42500000000000016
found solution for  10
------- 

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  198 , total integrated cost =  50.61388556890252
Improved over  198  iterations in  17.43631137162447  seconds by  99.83370514693911  percent.
Problem in initial value trasfer:  Vmean_exc -63.007617934351956 -63.00924393943926
weight =  6035.187506529953
set cost params:  1.0 0.0 6035.187506529953
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  30287.121479598834
Gradient descend method:  None
RUN  1 , total integrated cost =  29245.297194979954
RUN  2 , total integrated cost =  29228.743759826222
RUN  3 , total integrated cost =  29228.018407454565
RUN  4 , total integrated cost =  29227.22972891512
RUN  5 , total integrated cost =  29226.665747683688
RUN  6 , total integrated cost =  29225.968546700056
RUN  7 , total integrated cost =  29225.463122671044
RUN  8 , total integrated cost =  29224.79834354574
RUN  9 , total integrated cost =  29224.280802756883
RUN  10 , total integrated cost =  29223.59303366957
RUN  11 , tot

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  465 , total integrated cost =  26436.364986712637
Improved over  465  iterations in  37.030031433328986  seconds by  12.714171254207955  percent.
Problem in initial value trasfer:  Vmean_exc -56.67607513328868 -56.67896959657809
-------  40 0.5250000000000001 0.5500000000000003
found solution for  40
-------  45 0.5000000000000002 0.5750000000000003
found solution for  45
-------  50 0.47500000000000014 0.6000000000000003
found solution for  50
-------  55 0.4250000000000001 0.6250000000000003
found solution for  55
-------  60 0.5500000000000003 0.6250000000000003
found solution for  60
-------  65 0.5000000000000002 0.6500000000000004
found solution for  65
-------  70 0.4500000000000001 0.6750000000000004
found solution for  70
-------  75 0.5750000000000002 0.6750000000000004
[0, 5, 10, 15, 20, 25, 30, 40, 45, 50, 55, 60, 65, 70] []
closest index  60
set cost params:  1.0 0.0 10.0
precision vars =  [0]
interpolate adjoint :  True True True
RUN 

...

Control only changes marginally.
RUN  104 , total integrated cost =  32352.57990788199
Improved over  104  iterations in  6.962502345442772  seconds by  3.4977411170104062  percent.
Problem in initial value trasfer:  Vmean_exc -57.3451065462477 -57.32532670101387
-------  110 0.5000000000000002 0.8000000000000005
-------  115 0.4250000000000001 0.8250000000000005
-------  120 0.5500000000000003 0.8250000000000005
-------  125 0.47500000000000014 0.8500000000000005
-------  130 0.6000000000000003 0.8500000000000005
[0, 5, 10, 15, 20, 25, 30, 40, 45, 50, 55, 60, 65, 70, 80, 85, 95, 100, 110, 115, 120, 125, 135, 140, 145] [120, 135, 110, 95, 125, 140, 80, 100, 85, 115, 65, 60, 70, 50, 145, 55, 45, 40, 30, 25, 20, 15, 10, 5]
closest index  0
set cost params:  1.0 0.0 10.0
precision vars =  [0]
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  38696.59670609856
Gradient descend method:  None
RUN  1 , total integrated cost =  227.61870936857855
RUN  2 , total integrated

ERROR:root:Problem in initial value trasfer


RUN  300 , total integrated cost =  59.74209235671122
Control only changes marginally.
RUN  300 , total integrated cost =  59.74209235671122
Improved over  300  iterations in  19.206495644524693  seconds by  99.84561409156869  percent.
Problem in initial value trasfer:  Vmean_exc -63.51268191682144 -63.518470582011574
weight =  6482.4238465833105
set cost params:  1.0 0.0 6482.4238465833105
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  38292.59678572283
Gradient descend method:  None
RUN  1 , total integrated cost =  36959.7370362404
RUN  2 , total integrated cost =  36949.11105584641
RUN  3 , total integrated cost =  36947.7461948518
RUN  4 , total integrated cost =  36946.49442289824
RUN  5 , total integrated cost =  36944.5781538173
RUN  6 , total integrated cost =  36942.595795210334
RUN  7 , total integrated cost =  36940.407415031776
RUN  8 , total integrated cost =  36938.32610701876
RUN  9 , total integrated cost =  36933.203948124945
RUN  10 , total i

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  848 , total integrated cost =  33647.07629316216
Improved over  848  iterations in  55.84392672404647  seconds by  12.131641315829299  percent.
Problem in initial value trasfer:  Vmean_exc -56.69338138428776 -56.69573513557009
-------  135 0.5250000000000001 0.8750000000000006
-------  140 0.4500000000000001 0.9000000000000006
-------  145 0.5750000000000002 0.9000000000000006
------------------------------------------------------------
-------------------- 25
------------------------------------------------------------
found solution:  [0, 5, 10, 15, 20, 25, 30, 40, 45, 50, 55, 60, 65, 70, 80, 85, 95, 100, 110, 115, 120, 125, 135, 140, 145]
no solution:  []
-------  0 0.4000000000000001 0.3500000000000001
-------  5 0.4000000000000001 0.40000000000000013
-------  10 0.4250000000000001 0.42500000000000016
-------  15 0.4500000000000001 0.4500000000000002
-------  20 0.4500000000000001 0.4750000000000002
-------  25 0.4250000000000001 0.500000000000

In [17]:
factor_iteration = 20
full_converge = False
conv_init = [[False]*2] * len(exc)

for i in range(len(conv_init)):
    if i not in i_range:
        conv_init[i] = [True, True]
        
counter = 0

while full_converge == False:
    
    if counter > 100:
        break
        
    print(conv_init[::i_stepsize])
    full_converge = True
    
    for conv in conv_init[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range:        

        print("------- ", i, exc[i], inh[i])
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
        
        j = 1
        while cost_init[i][-j] == 0.:
            j += 1
            
        if j == cost_init[i].shape[0]-1:
            print("converged for ", i)
            if conv_init[i][0]:
                conv_init[i] = [True, True]
            else:
                conv_init[i] = [True, False]
            continue
    
        print("no convergence")
            
        weight_ = (factor_we * weights_init[i][1] * cost_uncontrolled[i] / cost_init[i][-j]
                   + factor_ws * weights_init[i][2] * cost_uncontrolled[i] / cost_init[i][-j]) - 1
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int( 500 * factor_iteration )

        weights_init[i] = cost.getParams()

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
        counter += 1

[[False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False]]
-------  0 0.4000000000000001 0.3500000000000001
converged for  0
-------  5 0.4000000000000001 0.40000000000000013
no convergence
weight =  8737.756947034784
set cost params:  1.0 0.0 8737.756947034784
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5096.673111058013
Gradient descend method:  None
RUN  1 , total integrated cost =  5096.672991228656
RUN  2 , total integrated cost =  5096.6729910322665
RUN  3 , total integrated cost =  5096.672991024078
RUN  4 , total integrated cost =  5096

ERROR:root:Problem in initial value trasfer


RUN  7 , total integrated cost =  5096.67299102399
RUN  8 , total integrated cost =  5096.67299102399
Control only changes marginally.
RUN  8 , total integrated cost =  5096.67299102399
Improved over  8  iterations in  0.8132188729941845  seconds by  2.3551446304281853e-06  percent.
Problem in initial value trasfer:  Vmean_exc -66.84502826732631 -66.86675362730958
-------  10 0.4250000000000001 0.42500000000000016
converged for  10
-------  15 0.4500000000000001 0.4500000000000002
no convergence
weight =  5847.534020654111
set cost params:  1.0 0.0 5847.534020654111
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  13015.460715666513
Gradient descend method:  None
RUN  1 , total integrated cost =  13015.458907027294
RUN  2 , total integrated cost =  13015.458801555187
RUN  3 , total integrated cost =  13015.458766743377
RUN  4 , total integrated cost =  13015.458765566967
RUN  5 , total integrated cost =  13015.458765566962


ERROR:root:Problem in initial value trasfer


RUN  6 , total integrated cost =  13015.458765566962
Control only changes marginally.
RUN  6 , total integrated cost =  13015.458765566962
Improved over  6  iterations in  0.5568036269396544  seconds by  1.4982946765940142e-05  percent.
Problem in initial value trasfer:  Vmean_exc -62.74509532337973 -62.778518940897044
-------  20 0.4500000000000001 0.4750000000000002
converged for  20
-------  25 0.4250000000000001 0.5000000000000002
no convergence
weight =  6598.883252094912
set cost params:  1.0 0.0 6598.883252094912
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  8230.52389741933
Gradient descend method:  None
RUN  1 , total integrated cost =  8230.523641013277
RUN  2 , total integrated cost =  8230.52363386345
RUN  3 , total integrated cost =  8230.523633552142
RUN  4 , total integrated cost =  8230.523633506622
RUN  5 , total integrated cost =  8230.52363350032
RUN  6 , total integrated cost =  8230.523633499364
RUN  7 , total integrated cost =  8230.52363

...

RUN  5 , total integrated cost =  34445.36450857015
RUN  6 , total integrated cost =  34445.36450857015
Control only changes marginally.
RUN  6 , total integrated cost =  34445.36450857015
Improved over  6  iterations in  0.5834086313843727  seconds by  0.00038325071672318245  percent.
Problem in initial value trasfer:  Vmean_exc -56.70390736011335 -56.703847087406935
-------  80 0.5250000000000001 0.7000000000000004
converged for  80
-------  85 0.47500000000000014 0.7250000000000004
converged for  85
-------  90 0.6000000000000003 0.7250000000000004
no convergence
weight =  8938.882324061093
set cost params:  1.0 0.0 8938.882324061093
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  39282.2030662182
Gradient descend method:  None
RUN  1 , total integrated cost =  39282.12662623461
RUN  2 , total integrated cost =  39282.1266262346


ERROR:root:Problem in initial value trasfer


RUN  3 , total integrated cost =  39282.1266262346
Control only changes marginally.
RUN  3 , total integrated cost =  39282.1266262346
Improved over  3  iterations in  0.3783098869025707  seconds by  0.00019459189563519885  percent.
Problem in initial value trasfer:  Vmean_exc -56.7014262420106 -56.70125736923931
-------  95 0.5250000000000001 0.7500000000000004
converged for  95
-------  100 0.4500000000000001 0.7750000000000005
converged for  100
-------  105 0.5750000000000002 0.7750000000000005
converged for  105
-------  110 0.5000000000000002 0.8000000000000005
converged for  110
-------  115 0.4250000000000001 0.8250000000000005
converged for  115
-------  120 0.5500000000000003 0.8250000000000005
converged for  120
-------  125 0.47500000000000014 0.8500000000000005
converged for  125
-------  130 0.6000000000000003 0.8500000000000005
no convergence
weight =  8906.49891433956
set cost params:  1.0 0.0 8906.49891433956
interpolate adjoint :  True True True
RUN  0 , total integra

ERROR:root:Problem in initial value trasfer


RUN  4 , total integrated cost =  38669.97043416481
Control only changes marginally.
RUN  4 , total integrated cost =  38669.97043416481
Improved over  4  iterations in  0.43670531176030636  seconds by  0.000192877204767683  percent.
Problem in initial value trasfer:  Vmean_exc -56.701855166499364 -56.701704816396315
-------  135 0.5250000000000001 0.8750000000000006
converged for  135
-------  140 0.4500000000000001 0.9000000000000006
converged for  140
-------  145 0.5750000000000002 0.9000000000000006
converged for  145


In [18]:
print(conv_init[::i_stepsize])

with open(init_file,'wb') as f:
    pickle.dump([bestControl_init, bestState_init, cost_init, runtime_init, grad_init, phi_init,
                 costnode_init, weights_init], f)

[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True]]


In [19]:
if os.path.isfile(final_file) :
    print("file found")
    
    with open(final_file,'rb') as f:
        load_array = pickle.load(f)

    bestControl_0 = load_array[0]
    bestState_0 = load_array[1]
    cost_0 = load_array[2]
    runtime_0 = load_array[3]
    grad_0 = load_array[4]
    phi_0 = load_array[5]
    costnode_0 = load_array[6]
    weights_0 = load_array[7]

file found


In [20]:
i_range_0 = range(0, len(exc),i_stepsize)
i_range_ = []

for i in i_range_0:
    if type(bestControl_0[i]) == type(None):
        i_range_.append(i)

i_range_0 = np.array(i_range_)
        
print(i_range_0)

[  5  15  25  45  65  75  85  95 115 125 135 145]


In [21]:
factor_iteration = 20
    
for i in i_range_0:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

# exc and inh control current 

    setinit(initVars[i], aln)
    aln.params.duration = dur
    control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1].copy()
    #control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]
 
    cost.setParams(1.0, 0.0, 10.0)

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 50 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_0[i][-j] == 0.:
        j += 1
    
    weight_ = 100
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = "HS"
    max_it = 50 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_0[i][-j] == 0.:
        j += 1
    
    weight_ = 100 * cost_uncontrolled[i] / cost_0[i][-j] - 1
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 50 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
        prec_variables_ = prec_vars, transition_time_ = trans_time)

-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  11.168248300175641
Gradient descend method:  None
RUN  1 , total integrated cost =  5.871040302245737
RUN  2 , total integrated cost =  5.870519792306232
RUN  3 , total integrated cost =  5.870516838173907
RUN  4 , total integrated cost =  5.8705166762113485
RUN  5 , total integrated cost =  5.870516642477097
RUN  6 , total integrated cost =  5.870516637306778
RUN  7 , total integrated cost =  5.870516633427575
RUN  8 , total integrated cost =  5.870516633427571
RUN  9 , total integrated cost =  5.87051663342757
State only changes marginally.


ERROR:root:Problem in initial value trasfer


RUN  10 , total integrated cost =  5.87051663342757
Control only changes marginally.
RUN  10 , total integrated cost =  5.87051663342757
Improved over  10  iterations in  2.810221226885915  seconds by  47.43565440486092  percent.
Problem in initial value trasfer:  Vmean_exc -67.89107350902081 -67.89412588197376
weight =  100
set cost params:  1.0 0.0 100.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  58.64963212916297
Gradient descend method:  HS
RUN  1 , total integrated cost =  58.57526312893253
RUN  2 , total integrated cost =  58.54621928992817
RUN  3 , total integrated cost =  58.52002745517044
RUN  4 , total integrated cost =  58.49781245440377
RUN  5 , total integrated cost =  58.478876510209844
RUN  6 , total integrated cost =  58.47001512262102
RUN  7 , total integrated cost =  58.40730594715956
RUN  8 , total integrated cost =  58.40611400082388
RUN  9 , total integrated cost =  58.383784057563325
RUN  10 , total integrated cost =  58.3662907721659


ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  34 , total integrated cost =  58.257842636457056
Improved over  34  iterations in  10.10025915876031  seconds by  0.6680169652949957  percent.
Problem in initial value trasfer:  Vmean_exc -67.81400797856165 -67.8176556034386
weight =  8748.5341357009
set cost params:  1.0 0.0 8748.5341357009
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5094.155958247446
Gradient descend method:  None
RUN  1 , total integrated cost =  5085.412876132629
RUN  2 , total integrated cost =  5085.4092905730095
RUN  3 , total integrated cost =  5085.409264640715
RUN  4 , total integrated cost =  5085.409264552774
RUN  5 , total integrated cost =  5085.409264498512
RUN  6 , total integrated cost =  5085.409264449502
RUN  7 , total integrated cost =  5085.409264400444
RUN  8 , total integrated cost =  5085.409264345075
RUN  9 , total integrated cost =  5085.409264311211
RUN  10 , total integrated cost =  5085.409264309503
RUN  11 , total integrated

...

Control only changes marginally.
RUN  57 , total integrated cost =  51.739418926594894
Improved over  57  iterations in  16.700164277106524  seconds by  48.29389667372826  percent.
Problem in initial value trasfer:  Vmean_exc -65.503602997306 -65.51784026672242
weight =  100
set cost params:  1.0 0.0 100.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  515.1188656042864
Gradient descend method:  HS
RUN  1 , total integrated cost =  512.3232457026826
RUN  2 , total integrated cost =  512.3043387826903
RUN  3 , total integrated cost =  512.3038290127955
RUN  4 , total integrated cost =  512.303613131978
RUN  5 , total integrated cost =  512.3036131319774
RUN  6 , total integrated cost =  512.3036131319772


ERROR:root:Problem in initial value trasfer


RUN  7 , total integrated cost =  512.3036131319772
Control only changes marginally.
RUN  7 , total integrated cost =  512.3036131319772
Improved over  7  iterations in  2.447489622980356  seconds by  0.5465248237427005  percent.
Problem in initial value trasfer:  Vmean_exc -63.54237212428925 -63.55948100504421
weight =  6497.109834392618
set cost params:  1.0 0.0 6497.109834392618
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  33163.87352982204
Gradient descend method:  None
RUN  1 , total integrated cost =  32937.90414675801
RUN  2 , total integrated cost =  32937.74373860363
RUN  3 , total integrated cost =  32937.73323684356
RUN  4 , total integrated cost =  32937.68670609732
RUN  5 , total integrated cost =  32937.653317739125
RUN  6 , total integrated cost =  32937.54713937861
RUN  7 , total integrated cost =  32937.42239941974
RUN  8 , total integrated cost =  32937.41291306493
RUN  9 , total integrated cost =  32937.37079178927
RUN  10 , total integrate

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  123 , total integrated cost =  32933.77700308034
Improved over  123  iterations in  39.78090365976095  seconds by  0.6938168019932789  percent.
Problem in initial value trasfer:  Vmean_exc -58.41208800723475 -58.3983898573531


In [22]:
"""
for i in i_range_0:
    
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    plotFunc.plot_control_current(aln, [bestControl_init[i], bestControl_0[i]],
        [costnode_init[i], costnode_0[i]], [weights_init[i], weights_0[i]], dur,
        dur_pre, dur_post, initVars[i], target[i], '', filename_ = '', transition_time_ = trans_time,
        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)
"""

'\nfor i in i_range_0:\n    \n    aln.params.mue_ext_mean = exc[i] * 5.\n    aln.params.mui_ext_mean = inh[i] * 5.\n\n    plotFunc.plot_control_current(aln, [bestControl_init[i], bestControl_0[i]],\n        [costnode_init[i], costnode_0[i]], [weights_init[i], weights_0[i]], dur,\n        dur_pre, dur_post, initVars[i], target[i], \'\', filename_ = \'\', transition_time_ = trans_time,\n        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)\n'

In [23]:
factor_iteration = 20
full_converge = False
conv_0 = [[False]*2] * len(exc)

for i in range(len(conv_0)):
    if i not in i_range_0:
        conv_0[i] = [True, True]

counter = 0

while full_converge == False:
    print('---------------', counter)
    
    if counter > 100:
        break
    
    print(conv_0[::i_stepsize])
    full_converge = True
    
    for conv in conv_0[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range_0:

        print("------- ", i, exc[i], inh[i])
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.

        j = 1
        while cost_0[i][-j] == 0.:
            j += 1
            
        if j == cost_0[i].shape[0]-1:
            print("converged for ", i)
            if conv_0[i][0]:
                conv_0[i] = [True, True]
            else:
                conv_0[i] = [True, False]
            continue
    
        print("no convergence")

        weight_ = (factor_we * weights_0[i][1] * cost_uncontrolled[i] / cost_0[i][-j]
                       + factor_ws * weights_0[i][2] * cost_uncontrolled[i] / cost_0[i][-j]) - 1
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = 500 * factor_iteration

        weights_0[i] = cost.getParams()

        bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
    counter += 1
    

--------------- 0
[[True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False]]
-------  5 0.4000000000000001 0.40000000000000013
no convergence
weight =  8767.972514077735
set cost params:  1.0 0.0 8767.972514077735
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5096.702502824831
Gradient descend method:  None
RUN  1 , total integrated cost =  5096.702501262058
RUN  2 , total integrated cost =  5096.7025012150025
RUN  3 , total integrated cost =  5096.702501214726
RUN  4 , total integrated cost =  5096.702501214718
RUN  5 , total integrated cost =  5096.702501214714
RUN  6 , total int

ERROR:root:Problem in initial value trasfer


RUN  7 , total integrated cost =  5096.702501214713
Control only changes marginally.
RUN  7 , total integrated cost =  5096.702501214713
Improved over  7  iterations in  2.7642492204904556  seconds by  3.159136952035624e-08  percent.
Problem in initial value trasfer:  Vmean_exc -67.17630597915132 -67.19194380278842
-------  15 0.4500000000000001 0.4500000000000002
no convergence
weight =  5855.5495959917735
set cost params:  1.0 0.0 5855.5495959917735
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  13015.745852123731
Gradient descend method:  None
RUN  1 , total integrated cost =  13015.745714777342
RUN  2 , total integrated cost =  13015.745705831163
RUN  3 , total integrated cost =  13015.745704520634
RUN  4 , total integrated cost =  13015.745704520621


ERROR:root:Problem in initial value trasfer


RUN  5 , total integrated cost =  13015.745704520621
Control only changes marginally.
RUN  5 , total integrated cost =  13015.745704520621
Improved over  5  iterations in  1.9798318557441235  seconds by  1.1340349743704792e-06  percent.
Problem in initial value trasfer:  Vmean_exc -63.530380365803694 -63.56388178296811
-------  25 0.4250000000000001 0.5000000000000002
no convergence
weight =  6617.435709943068
set cost params:  1.0 0.0 6617.435709943068
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  8230.63487332028
Gradient descend method:  None
RUN  1 , total integrated cost =  8230.634854592341
RUN  2 , total integrated cost =  8230.634854344615
RUN  3 , total integrated cost =  8230.63485434461


...

RUN  4 , total integrated cost =  34490.91844671141
Control only changes marginally.
RUN  4 , total integrated cost =  34490.91844671141
Improved over  4  iterations in  1.532892806455493  seconds by  1.267823535044954e-07  percent.
Problem in initial value trasfer:  Vmean_exc -56.703439376128635 -56.70341052230032
-------  85 0.47500000000000014 0.7250000000000004
converged for  85
-------  95 0.5250000000000001 0.7500000000000004
converged for  95
-------  115 0.4250000000000001 0.8250000000000005
converged for  115
-------  125 0.47500000000000014 0.8500000000000005
converged for  125
-------  135 0.5250000000000001 0.8750000000000006
converged for  135
-------  145 0.5750000000000002 0.9000000000000006
converged for  145
--------------- 99
[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [False, False], [True, True], [True

ERROR:root:Problem in initial value trasfer


RUN  3 , total integrated cost =  34490.951754923284
Control only changes marginally.
RUN  3 , total integrated cost =  34490.951754923284
Improved over  3  iterations in  1.2549759484827518  seconds by  8.635107917598361e-08  percent.
Problem in initial value trasfer:  Vmean_exc -56.70343883240097 -56.70341002953137
-------  85 0.47500000000000014 0.7250000000000004
converged for  85
-------  95 0.5250000000000001 0.7500000000000004
converged for  95
-------  115 0.4250000000000001 0.8250000000000005
converged for  115
-------  125 0.47500000000000014 0.8500000000000005
converged for  125
-------  135 0.5250000000000001 0.8750000000000006
converged for  135
-------  145 0.5750000000000002 0.9000000000000006
converged for  145
--------------- 100
[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [False, False], [True, True], [T

ERROR:root:Problem in initial value trasfer


RUN  3 , total integrated cost =  34490.98386985352
Control only changes marginally.
RUN  3 , total integrated cost =  34490.98386985352
Improved over  3  iterations in  1.2393164616078138  seconds by  1.215059484138692e-07  percent.
Problem in initial value trasfer:  Vmean_exc -56.70343810776088 -56.7034093727998
-------  85 0.47500000000000014 0.7250000000000004
converged for  85
-------  95 0.5250000000000001 0.7500000000000004
converged for  95
-------  115 0.4250000000000001 0.8250000000000005
converged for  115
-------  125 0.47500000000000014 0.8500000000000005
converged for  125
-------  135 0.5250000000000001 0.8750000000000006
converged for  135
-------  145 0.5750000000000002 0.9000000000000006
converged for  145
--------------- 101


In [24]:
print(conv_0[::i_stepsize])

with open(final_file,'wb') as f:
    pickle.dump([bestControl_0, bestState_0, cost_0, runtime_0, grad_0, phi_0,
                 costnode_0, weights_0], f)

[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True]]


In [25]:
bestControl_1 = [None] * len(exc)
bestState_1 = [None] * len(exc)
cost_1 = [None] * len(exc)
runtime_1 = [None] * len(exc)
grad_1 = [None] * len(exc)
phi_1 = [None] * len(exc)
costnode_1 = [None] * len(exc)
weights_1 = [None] * len(exc)

conv_1 = [[False]*2] * len(exc)

In [26]:
if os.path.isfile(final_file_1) :
    print("file found")
    
    with open(final_file_1,'rb') as f:
        load_array = pickle.load(f)

    bestControl_1 = load_array[0]
    bestState_1 = load_array[1]
    cost_1 = load_array[2]
    runtime_1 = load_array[3]
    grad_1 = load_array[4]
    phi_1 = load_array[5]
    costnode_1 = load_array[6]
    weights_1 = load_array[7]

file found


In [27]:
i_range_1 = range(0, len(exc),i_stepsize)
i_range_ = []

for i in i_range_1:
    if type(bestControl_1[i]) == type(None):
        i_range_.append(i)

i_range_1 = np.array(i_range_)  

print(i_range_1)

[  5  15  25  45  65  75  85  95 115 125 135 145]


In [28]:
factor_iteration = 20

for i in i_range_1:        

    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    cost.setParams(1.0, 1. * factor_we, 1. * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int( 500 * factor_iteration )

    weights_1[i] = cost.getParams()

    bestControl_1[i], bestState_1[i], cost_1[i], runtime_1[i], grad_1[i], phi_1[i], costnode_1[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)

-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  3.2999343288698832
Gradient descend method:  None
RUN  1 , total integrated cost =  0.58429833449758
RUN  2 , total integrated cost =  0.5842976681938129
RUN  3 , total integrated cost =  0.584297668193812
RUN  4 , total integrated cost =  0.5842976681938115
RUN  5 , total integrated cost =  0.5842976681938113


ERROR:root:Problem in initial value trasfer


RUN  6 , total integrated cost =  0.5842976681938113
Control only changes marginally.
RUN  6 , total integrated cost =  0.5842976681938113
Improved over  6  iterations in  0.5512914676219225  seconds by  82.29365769245736  percent.
Problem in initial value trasfer:  Vmean_exc -67.99767268194225 -68.00013606709491
-------  15 0.4500000000000001 0.4500000000000002
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  19.21842862220188
Gradient descend method:  None
RUN  1 , total integrated cost =  2.260674811114141
RUN  2 , total integrated cost =  2.2606022492262405
RUN  3 , total integrated cost =  2.2606006481377556
RUN  4 , total integrated cost =  2.2606001071529995
RUN  5 , total integrated cost =  2.2605936255332137
RUN  6 , total integrated cost =  2.2605664084986294
RUN  7 , total integrated cost =  2.2605627650996327
RUN  8 , total integrated cost =  2.260562357194412
RUN  9 , total integrated cost =  2.260559602496466
RUN  10 , 

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  77 , total integrated cost =  2.2588656389793735
Improved over  77  iterations in  5.004083842039108  seconds by  88.24635622722118  percent.
Problem in initial value trasfer:  Vmean_exc -67.5719679575557 -67.57850040365739
-------  25 0.4250000000000001 0.5000000000000002
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  8.8344634995087
Gradient descend method:  None
RUN  1 , total integrated cost =  1.2541010781448267
RUN  2 , total integrated cost =  1.253950905530462
RUN  3 , total integrated cost =  1.2539508992293906
RUN  4 , total integrated cost =  1.2539508991278134
RUN  5 , total integrated cost =  1.253950899127812


...

Control only changes marginally.
RUN  73 , total integrated cost =  2.4186440032161536
Improved over  73  iterations in  4.755160443484783  seconds by  90.74476229531457  percent.
Problem in initial value trasfer:  Vmean_exc -72.55899593927575 -72.58637922531625
-------  135 0.5250000000000001 0.8750000000000006
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  48.38327814375877
Gradient descend method:  None
RUN  1 , total integrated cost =  3.9048043259271212
RUN  2 , total integrated cost =  3.9042198325737636
RUN  3 , total integrated cost =  3.9042035998577154
RUN  4 , total integrated cost =  3.9041946711905684
RUN  5 , total integrated cost =  3.9040218885944378
RUN  6 , total integrated cost =  3.9039399158362698
RUN  7 , total integrated cost =  3.90393016042297
RUN  8 , total integrated cost =  3.8999212681721196
RUN  9 , total integrated cost =  3.899796720320706
RUN  10 , total integrated cost =  3.8997943251206557
RUN  11

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  26 , total integrated cost =  3.8915121496114633
Improved over  26  iterations in  1.8588736150413752  seconds by  91.95690680972709  percent.
Problem in initial value trasfer:  Vmean_exc -69.38039057558228 -69.40049179219862
-------  145 0.5750000000000002 0.9000000000000006
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  54.530058018165185
Gradient descend method:  None
RUN  1 , total integrated cost =  5.241093078857367
RUN  2 , total integrated cost =  5.240863476966269
RUN  3 , total integrated cost =  5.240103581772415
RUN  4 , total integrated cost =  5.239283548854763
RUN  5 , total integrated cost =  5.239232889130273
RUN  6 , total integrated cost =  5.236942972552228
RUN  7 , total integrated cost =  5.23578690568905
RUN  8 , total integrated cost =  5.235781098359311
RUN  9 , total integrated cost =  5.235757264141597
RUN  10 , total integrated cost =  5.235621252739497
RUN  11 , tot

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  33 , total integrated cost =  5.234134040679781
Improved over  33  iterations in  2.225814374163747  seconds by  90.40137819230603  percent.
Problem in initial value trasfer:  Vmean_exc -66.59015986987794 -66.6003671982017


In [29]:
factor_iteration = 20
full_converge = False

for i in range(len(conv_1)):
    if i not in i_range_1:
        conv_1[i] = [True, True]
        
counter = 0

while full_converge == False:
    
    print('---------------', counter)
    
    print(conv_1[::i_stepsize])
    full_converge = True
    
    for conv in conv_1[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range_1:        

        print("------- ", i, exc[i], inh[i])
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
        
        j = 1
        while cost_1[i][-j] == 0.:
            j += 1
            
        if j == cost_1[i].shape[0]-1:
            print("converged for ", i)
            if conv_1[i][0]:
                conv_1[i] = [True, True]
            else:
                conv_1[i] = [True, False]
            continue
    
        print("no convergence")
        
        cost.setParams(1.0, 1. * factor_we, 1. * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_1[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int( 500 * factor_iteration )

        weights_1[i] = cost.getParams()

        bestControl_1[i], bestState_1[i], cost_1[i], runtime_1[i], grad_1[i], phi_1[i], costnode_1[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
        counter += 1

ERROR:root:Problem in initial value trasfer


--------------- 0
[[True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False]]
-------  5 0.4000000000000001 0.40000000000000013
no convergence
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  0.5842976681938113
Gradient descend method:  None
RUN  1 , total integrated cost =  0.5842976681938113
Control only changes marginally.
RUN  1 , total integrated cost =  0.5842976681938113
Improved over  1  iterations in  0.1536941695958376  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -67.99767268194225 -68.00013606709491
-------  15 0.45000

ERROR:root:Problem in initial value trasfer
ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  2.2588656389793735
Control only changes marginally.
RUN  1 , total integrated cost =  2.2588656389793735
Improved over  1  iterations in  0.1556549295783043  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -67.5719679575557 -67.57850040365739
-------  25 0.4250000000000001 0.5000000000000002
no convergence
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  1.253950899127811
Gradient descend method:  None
RUN  1 , total integrated cost =  1.253950899127811
Control only changes marginally.
RUN  1 , total integrated cost =  1.253950899127811
Improved over  1  iterations in  0.15374137833714485  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -71.4250556652635 -71.44272259061798
-------  45 0.5000000000000002 0.5750000000000003
no convergence
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  3.517515991884757


ERROR:root:Problem in initial value trasfer
ERROR:root:Problem in initial value trasfer


...

RUN  1 , total integrated cost =  4.003387021900956
Control only changes marginally.
RUN  1 , total integrated cost =  4.003387021900956
Improved over  1  iterations in  0.15687363594770432  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -68.70152979478874 -68.71912646761791
-------  115 0.4250000000000001 0.8250000000000005
no convergence
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  0.6471609245785269
Gradient descend method:  None
RUN  1 , total integrated cost =  0.6471609245785269
Control only changes marginally.
RUN  1 , total integrated cost =  0.6471609245785269
Improved over  1  iterations in  0.1547650694847107  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -76.1056531200039 -76.13846591121562
-------  125 0.47500000000000014 0.8500000000000005
no convergence
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  2.41864400321

ERROR:root:Problem in initial value trasfer
ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  2.4186440032161536
Control only changes marginally.
RUN  1 , total integrated cost =  2.4186440032161536
Improved over  1  iterations in  0.15766539424657822  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -72.55899593927575 -72.58637922531625
-------  135 0.5250000000000001 0.8750000000000006
no convergence
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  3.8915121496114633
Gradient descend method:  None
RUN  1 , total integrated cost =  3.8915121496114633
Control only changes marginally.
RUN  1 , total integrated cost =  3.8915121496114633
Improved over  1  iterations in  0.15364466793835163  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -69.38039057558228 -69.40049179219862
-------  145 0.5750000000000002 0.9000000000000006
no convergence
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5.23413404

ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  5.234134040679781
Control only changes marginally.
RUN  1 , total integrated cost =  5.234134040679781
Improved over  1  iterations in  0.15511517971754074  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -66.59015986987794 -66.6003671982017
--------------- 12
[[True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False]]
-------  5 0.4000000000000001 0.40000000000000013
converged for  5
-------  15 0.4500000000000001 0.4500000000000002
converged for  15
-------  25 0.4250000000000001 0.5000000000000002
converged for  25
-------  45 0.5000000000000002 0.5

In [30]:
print(conv_1[::i_stepsize])

with open(final_file_1,'wb') as f:
    pickle.dump([bestControl_1, bestState_1, cost_1, runtime_1, grad_1, phi_1,
                 costnode_1, weights_1], f)

[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True]]
